# Sentinel-2 binary segmentation — reproducible companion

This is a **newly organized walkthrough** based on the preserved original lab (`original_lab3.ipynb`), not a replacement for the student’s submitted work. It requires the original `data/sentinel2.npz` (not included).

**Default:** 35 training, 5 validation, 10 independently evaluated image patches. The archived lab instead used 40 training / 10 reused for validation and final reporting.

## 1. Imports and load dataset

Run this notebook from the repository root or `notebooks/` directory. Install dependencies from `requirements.txt` first.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from sentinel_unet.data import load_dataset
from sentinel_unet.model import build_unet
from sentinel_unet.metrics import evaluate_masks
from sentinel_unet.visualization import save_examples, save_history

import tensorflow as tf

data = load_dataset(ROOT / "data" / "sentinel2.npz", mode="three-way")
print("Train/validation/test:", len(data.x_train), len(data.x_val), len(data.x_test))

## 2. Model architecture and training

The architecture matches the original two-level U-Net-inspired network. Training from scratch may be computationally expensive.

In [ ]:
tf.keras.utils.set_random_seed(42)
model = build_unet(tuple(data.x_train.shape[1:]))
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()
history = model.fit(data.x_train, data.y_train, validation_data=(data.x_val, data.y_val), epochs=100, batch_size=2)

## 3. Evaluate on the held-out 10 patches

Evaluate the **original pixel regions**, excluding artificial padding. Pixel accuracy alone does not measure positive-class segmentation quality.

In [ ]:
probs = model.predict(data.x_test)
metrics = evaluate_masks(data.y_test, probs, pad=data.pad)
print(metrics)
output = ROOT / "outputs"
output.mkdir(exist_ok=True)
save_examples(data.x_test, data.y_test, probs, output / "segmentation_examples.png", pad=data.pad)
save_history(history.history, output / "training_history.png")

## 4. Convert to TFLite and compare

The optimizer requests dynamic-range quantization. Inspect tensor data types before asserting that every operator stores 8-bit weights.

In [ ]:
from sentinel_unet.tflite_utils import convert_dynamic_range, predict_tflite, get_tflite_tensor_types

path = convert_dynamic_range(model, output / "unet_dynamic_range.tflite")
tflite_probs = predict_tflite(path, data.x_test)
print("Keras:", metrics)
print("TFLite:", evaluate_masks(data.y_test, tflite_probs, pad=data.pad))
print("Artifact details:", get_tflite_tensor_types(path))

## Limitations

These cells **have no saved execution results**. No dataset, trained weights or converted TFLite file were supplied, so the revised pipeline has not been end-to-end validated. See `docs/experiment_notes.md` for details on the original reported run.